In [ ]:
!pip install -q openai-whisper
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("*"):
    print(path)

In [ ]:
import os
import json
import shutil
import torch
import whisper

from IPython.display import display, FileLink

device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "large-v3"

model = whisper.load_model(MODEL_NAME, device=device)

print(f"Model: {MODEL_NAME}")
print(f"Device: {device}")

In [ ]:
AUDIO_DIR = "/kaggle/input/datasets/snnguyn1234/working_folder"

audio_files = sorted(
    os.path.join(AUDIO_DIR, filename)
    for filename in os.listdir(AUDIO_DIR)
    if filename.lower().endswith(".mp3")
)

output_folder_name = os.path.basename(AUDIO_DIR)
output_root = os.path.join("/kaggle/working", output_folder_name)

os.makedirs(output_root, exist_ok=True)

for audio_path in audio_files:
    audio_filename = os.path.basename(audio_path)
    audio_name = os.path.splitext(audio_filename)[0]

    print(f"\nTranscribing: {audio_filename}")

    lesson_folder = os.path.join(output_root, audio_name)
    os.makedirs(lesson_folder, exist_ok=True)

    shutil.copy2(
        audio_path,
        os.path.join(lesson_folder, "audio.mp3")
    )

    result = model.transcribe(
        audio_path,
        language="en",
        task="transcribe",
        fp16=(device == "cuda"),

        # Decoding
        temperature=0,
        beam_size=5,
        condition_on_previous_text=False,

        # Timing
        word_timestamps=True,

        # Hallucination detection
        hallucination_silence_threshold=2.0,

        verbose=False
    )

        
    timestamp_data = [
        {
            "start": round(segment["start"], 3),
            "end": round(segment["end"], 3),
            "text": segment["text"].strip(),
            "ts_idx": index
        }
        for index, segment in enumerate(result["segments"])
    ]


    with open(
        os.path.join(lesson_folder, "raw_timestamp.json"),
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(timestamp_data, f, ensure_ascii=False, indent=4)

    transcript = "\n".join(
        item["text"] for item in timestamp_data
    )

    with open(
        os.path.join(lesson_folder, "raw_text.txt"),
        "w",
        encoding="utf-8"
    ) as f:
        f.write(transcript)

    with open(
        os.path.join(lesson_folder, "text.txt"),
        "w",
        encoding="utf-8"
    ) as f:
        f.write("")

    print(transcript[:100])
    print(f"Saved to: {lesson_folder}")


zip_path = shutil.make_archive(
    output_root,
    "zip",
    output_root
)

print("\nFinished.")
display(FileLink(zip_path))